# 08 — Refresh Hiring Analytics Dashboard

**Jackson and Jackson HR Digital**

Final pipeline step: finds the deployed Lakeview dashboard and triggers a dataset refresh
so stakeholders see current data when the pipeline completes.

### What this notebook does
1. Looks up the `Jackson and Jackson HR Digital — Hiring Analytics` dashboard via the REST API
2. Triggers a published dashboard refresh
3. Prints the dashboard URL for stakeholders

### Prerequisites
- `databricks bundle deploy` has been run (dashboard is deployed to workspace)
- Notebooks `00` – `07` have completed successfully

**This is the final notebook in the pipeline.**

## Setup

Configure widgets, resolve workspace host and token, and set the target dashboard name.

In [ ]:
dbutils.widgets.text("catalog",        "bx4",      "UC Catalog")
dbutils.widgets.text("schema",         "hrd_2030", "UC Schema")
dbutils.widgets.text("warehouse_id",   "",         "SQL Warehouse ID")
dbutils.widgets.text("dashboard_name", "Jackson and Jackson HR Digital — Hiring Analytics", "Dashboard Display Name")

In [ ]:
import requests, time

catalog        = dbutils.widgets.get("catalog")
schema         = dbutils.widgets.get("schema")
warehouse_id   = dbutils.widgets.get("warehouse_id")
dashboard_name = dbutils.widgets.get("dashboard_name")

ctx   = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
host  = ctx.apiUrl().get().rstrip("/")
hdrs  = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Catalog        : {catalog}")
print(f"Schema         : {schema}")
print(f"Warehouse ID   : {warehouse_id or '(not set)'}")
print(f"Dashboard Name : {dashboard_name}")
print(f"Host           : {host}")

## Find Dashboard by Name

List all Lakeview dashboards via REST API and locate the Jackson and Jackson HR Digital dashboard by its display name.

In [ ]:
# ---------------------------------------------------------------------------
# Find the dashboard by display name via the Lakeview REST API
# ---------------------------------------------------------------------------
dashboard_id = None
dashboard_url = None

resp = requests.get(
    f"{host}/api/2.0/lakeview/dashboards",
    headers=hdrs,
    params={"page_size": 100},
    timeout=30,
)

if resp.status_code == 200:
    dashboards = resp.json().get("dashboards", [])
    for d in dashboards:
        if d.get("display_name", "") == dashboard_name:
            dashboard_id  = d["dashboard_id"]
            dashboard_url = f"{host}/dashboardsv3/{dashboard_id}"
            print(f"✅ Found dashboard: {dashboard_name}")
            print(f"   Dashboard ID  : {dashboard_id}")
            print(f"   Dashboard URL : {dashboard_url}")
            break

    if not dashboard_id:
        print(f"⚠️  Dashboard '{dashboard_name}' not found in workspace.")
        print(f"   Available dashboards ({len(dashboards)}):")
        for d in dashboards[:10]:
            print(f"     - {d.get('display_name')} ({d.get('dashboard_id')})")
else:
    print(f"⚠️  Could not list dashboards: HTTP {resp.status_code} — {resp.text[:300]}")

## Inject Catalog / Schema

If the dashboard definition contains `__CATALOG__`/`__SCHEMA__` placeholders, replace them with the runtime catalog and schema values.

In [ ]:
# ---------------------------------------------------------------------------
# Inject catalog/schema into dashboard definition (replace __CATALOG__/__SCHEMA__)
# This allows the dashboard JSON to ship with placeholders and be resolved at
# pipeline runtime, so the bundle file stays environment-agnostic.
# Uses the Databricks SDK so auth is handled automatically and failures raise.
# ---------------------------------------------------------------------------
if dashboard_id:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

    dash = w.lakeview.get(dashboard_id=dashboard_id)
    serialized = dash.serialized_dashboard or ""
    wh_id = dash.warehouse_id or warehouse_id

    if "__CATALOG__" in serialized or "__SCHEMA__" in serialized:
        patched = serialized.replace("__CATALOG__", catalog).replace("__SCHEMA__", schema)
        w.lakeview.update(
            dashboard_id=dashboard_id,
            display_name=dashboard_name,
            serialized_dashboard=patched,
            warehouse_id=wh_id,
        )
        print(f"✅ Dashboard draft updated: catalog={catalog}, schema={schema}")
    else:
        print("ℹ️  No placeholders found — skipping patch.")
else:
    print("Skipping catalog/schema injection — dashboard not found.")

## Trigger Dashboard Refresh

Publish a dataset refresh so the dashboard shows current data from the Gold table when the pipeline completes.

In [ ]:
# ---------------------------------------------------------------------------
# Publish the dashboard so current data is visible to stakeholders
# ---------------------------------------------------------------------------
if not dashboard_id:
    print("Skipping publish — dashboard not found.")
else:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    w.lakeview.publish(
        dashboard_id=dashboard_id,
        warehouse_id=warehouse_id,
        embed_credentials=True,
    )
    dashboard_url = f"{host}/sql/dashboardsv3/{dashboard_id}"
    print(f"✅ Dashboard published.")
    print(f"   URL: {dashboard_url}")

## Pipeline Complete

Print a summary of all deployed resources — Gold table, Genie Space, ML endpoint, Agent endpoint, and Dashboard URL.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline complete — print stakeholder summary
# ---------------------------------------------------------------------------
print("=" * 60)
print("Jackson and Jackson HR Digital — Pipeline Complete")
print("=" * 60)
print()
print(f"Gold table   : {catalog}.{schema}.candidate_scoring_summary")
print(f"Genie Space  : Create in UI if not auto-created")
print(f"ML Endpoint  : hire-right-model-endpoint")
print(f"Agent        : hire-right-agent-endpoint")
if dashboard_url:
    print(f"Dashboard    : {dashboard_url}")
else:
    print(f"Dashboard    : Deploy with 'databricks bundle deploy' then check workspace")
print()
print("✅ All pipeline tasks complete.")